# Reproduce the final evaluation

This clean notebook loads one verified checkpoint and evaluates it on the frozen RSNA test split and the zero-shot Kermany cohort. Install the repository with `pip install -e .` and create `configs/local.yaml` before running.

In [ ]:
from pathlib import Path
import json
import torch

from fair2026_lms_kan.config import load_config
from fair2026_lms_kan.data import LungCropCache, discover_kermany, load_manifest
from fair2026_lms_kan.evaluation import evaluate_datasets
from fair2026_lms_kan.models.classifier import build_model

In [ ]:
CONFIG_PATH = Path('../configs/local.yaml')
CHECKPOINT_PATH = Path('../runs/fastkan_crop_seed42/checkpoint.pt')
OUTPUT_DIR = Path('../runs/reproduced_fastkan_seed42')

config = load_config(CONFIG_PATH)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model_config = dict(config['model'])
metadata = checkpoint.get('metadata', {})
head = metadata.get('head') or model_config.pop('head')
model_config.pop('head', None)
model = build_model(head=head, **model_config).to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()

In [ ]:
paths = config['paths']
split_data = load_manifest(paths['manifest'], paths.get('rsna_root'))
datasets = {
    'RSNA_test': split_data['test'],
    'Kermany': discover_kermany(paths['kermany_root']),
}
crop_cache = None
if metadata.get('use_lung_crop', True):
    crop_cache = LungCropCache(
        paths['segmentation_checkpoint'], paths['crop_cache'], device
    )

In [ ]:
metrics = evaluate_datasets(
    model,
    datasets,
    OUTPUT_DIR,
    device,
    crop_cache=crop_cache,
    image_size=config['data']['image_size'],
    batch_size=config['training']['batch_size'],
    num_workers=config['training']['num_workers'],
)
print(json.dumps(metrics, indent=2))